In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import curve_fit

In [2]:
def estimate_sir_parameters(file_path, training_days_count, N, model, dt_fit):
    # Reads data and estimates optimal beta and gamma using curve_fit.
    df = pd.read_csv(file_path)
    days_data = df['day'].values[:training_days_count]
    infected_data = df['infected'].values[:training_days_count]
    
    I0 = infected_data[0]
    S0 = N - I0
    R0 = 0
    
    # Internal function to run the model for the optimizer
    def model_infected_wrapper(t, b, g):
        max_day = t[-1] + 1 
        t_sim, _, I_sim, _ = model(b, g, S0, I0, R0, max_day, dt_fit)
        return np.interp(t, t_sim, I_sim)

    # Optimize to find the best beta and gamma
    optimal_params, _ = curve_fit(
        model_infected_wrapper, 
        days_data, 
        infected_data, 
        bounds=(0, [3.0, 1.0]), 
        p0=[1.6, 0.4]
    )
    
    beta, gamma = optimal_params
    return beta, gamma, S0, I0, R0

In [3]:
def plot_sir_model(t, S, I, R, real_data=None, title=""):
    """
    Parameters:
        t (array-like): Array of time steps.
        S (array-like): Simulated susceptible population over time.
        I (array-like): Simulated infected population over time.
        R (array-like): Simulated recovered/removed population over time.
        real_data (tuple, optional): A tuple containing (real_t, real_I) arrays to overlay 
                                     observed data points. Defaults to None.
        title (str, optional): The title of the plot.
    """
    # Create the figure with specific dimensions
    plt.figure(figsize=(10, 6))
    
    # Plot the continuous Euler simulation curves
    plt.plot(t, S, label=r'Susceptible ($S$)', color='#1f77b4', linewidth=2.5)
    plt.plot(t, I, label=r'Infected ($I$)', color='#d62728', linewidth=2.5)
    plt.plot(t, R, label=r'Recovered ($R$)', color='#2ca02c', linewidth=2.5)
    
    # Optionally overlay real-world observed data points
    if real_data is not None:
        real_t, real_I = real_data
        plt.scatter(real_t, real_I, color='black', zorder=5, label='Observed Data ($I$)')
        
    # Formatting for academic presentation
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Time (Days)', fontsize=12)
    plt.ylabel('Population', fontsize=12)
    plt.legend(loc='best', fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout() 
    
    # Display the plot window
    plt.show()

In [4]:
def plot_model_sensitivity(solver_func, S0, I0, R0, days, dt, 
                           betas_to_test, fixed_gamma, 
                           gammas_to_test, fixed_beta, 
                           method_name="Model"):
    """
    Generates a parameter sensitivity plot for any compatible SIR numerical solver.
    
    Parameters:
        solver_func (callable): The numerical method function (e.g., solve_sir_euler)
        S0, I0, R0 (int): Initial population conditions
        days (int): Number of days to simulate
        dt (float): Time step size
        betas_to_test (list): Array of transmission rates to test
        fixed_gamma (float): The recovery rate to hold constant for the beta plot
        gammas_to_test (list): Array of recovery rates to test
        fixed_beta (float): The transmission rate to hold constant for the gamma plot
        method_name (str): Name of the method for the plot title (e.g., "RK4")
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # --- Plot 1: Varying Transmission Rate (Beta) ---
    colors_beta = ['#ff9999', '#ff4d4d', '#cc0000', '#800000']
    for i, b in enumerate(betas_to_test):
        # Call the passed solver function
        t, S, I, R = solver_func(b, fixed_gamma, S0, I0, R0, days, dt)
        R0_val = b / fixed_gamma
        ax1.plot(t, I, label=rf'$\beta = {b}$ ($R_0 = {R0_val:.1f}$)', 
                 color=colors_beta[i], linewidth=2.5)

    ax1.set_title(rf'{method_name}: Varying $\beta$ (fixed $\gamma={fixed_gamma}$)', 
                  fontsize=13, fontweight='bold')
    ax1.set_xlabel('Time (Days)', fontsize=12)
    ax1.set_ylabel('Infected Population ($I$)', fontsize=12)
    ax1.legend()
    ax1.grid(True, linestyle='--', alpha=0.7)

    # --- Plot 2: Varying Recovery Rate (Gamma) ---
    colors_gamma = ['#99ccff', '#3399ff', '#0066cc', '#003366']
    for i, g in enumerate(gammas_to_test):
        # Call the passed solver function
        t, S, I, R = solver_func(fixed_beta, g, S0, I0, R0, days, dt)
        R0_val = fixed_beta / g
        ax2.plot(t, I, label=rf'$\gamma = {g}$ ($R_0 = {R0_val:.1f}$)', 
                 color=colors_gamma[i], linewidth=2.5)

    ax2.set_title(rf'{method_name}: Varying $\gamma$ (fixed $\beta={fixed_beta}$)', 
                  fontsize=13, fontweight='bold')
    ax2.set_xlabel('Time (Days)', fontsize=12)
    ax2.legend()
    ax2.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

In [5]:
def run_and_plot_data_driven_model(csv_file, total_population, days_to_train, 
                                   simulation_days, time_step, solver_func, plot_title):
    """
    Runs the complete data-driven pipeline: estimates parameters from data,
    runs the specified numerical simulation, and plots the results.
    """
    # 1. Extract real data for the plot overlay
    full_data = pd.read_csv(csv_file)
    real_days = full_data['day'].values
    real_infected = full_data['infected'].values

    # 2. Calculate parameters from data
    print(f"--- Running Pipeline for {solver_func.__name__} ---")
    print("Calculating parameters from data...")
    calc_beta, calc_gamma, calc_S0, calc_I0, calc_R0 = estimate_sir_parameters(
        csv_file, days_to_train, total_population, solver_func, time_step
    )
    
    print(f"Calculated Transmission Rate (Beta): {calc_beta:.4f}")
    print(f"Calculated Recovery Rate (Gamma): {calc_gamma:.4f}")
    print(f"Calculated Basic Reproduction Number (R0): {calc_beta/calc_gamma:.2f}")

    # 3. Call the solver using the CALCULATED parameters
    print("Running simulation...")
    t, S, I, R = solver_func(
        beta=calc_beta, 
        gamma=calc_gamma, 
        S0=calc_S0, 
        I0=calc_I0, 
        R0=calc_R0, 
        days=simulation_days, 
        dt=time_step
    )

    # 4. Plot the results
    print("Generating plot...\n")
    plot_sir_model(
        t, S, I, R, 
        real_data=(real_days, real_infected), 
        title=plot_title
    )
    
    return calc_beta, calc_gamma

In [6]:
import numpy as np
import matplotlib.pyplot as plt

def plot_convergence_comparison(model1, model2, name1, name2, S0, I0, R0, beta, gamma, days):
    """
    Computes the maximum error for two numerical solvers across various time steps
    and plots the convergence on both a linear and log-log scale side-by-side.
    
    Parameters:
        model1 (callable): First numerical solver (e.g., solve_sir_euler)
        model2 (callable): Second numerical solver (e.g., solve_sir_rk4). 
                           This is also used to calculate the high-precision baseline.
        name1 (str): Name for the first model's legend label
        name2 (str): Name for the second model's legend label
        S0, I0, R0, beta, gamma, days: SIR model parameters
    """
    # 1. Generate a High-Precision Baseline (using the higher-order model2 with a tiny dt)
    dt_baseline = 0.001
    t_true, _, I_true, _ = model2(beta, gamma, S0, I0, R0, days, dt_baseline)

    # 2. Define the step sizes to test
    dt_values = np.array([2.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01])
    
    errors1 = []
    errors2 = []

    # 3. Calculate the maximum absolute error for each dt
    for dt in dt_values:
        # Call the passed solver functions
        t1, _, I1, _ = model1(beta, gamma, S0, I0, R0, days, dt)
        t2, _, I2, _ = model2(beta, gamma, S0, I0, R0, days, dt)
        
        # Interpolate the baseline curve to match the coarse time steps
        I_true_interp1 = np.interp(t1, t_true, I_true)
        I_true_interp2 = np.interp(t2, t_true, I_true)
        
        # Calculate maximum absolute error
        err1 = np.max(np.abs(I1 - I_true_interp1))
        err2 = np.max(np.abs(I2 - I_true_interp2))
        
        errors1.append(err1)
        errors2.append(err2)

    # 4. Calculate empirical convergence orders (slope of the log-log line)
    slope1 = np.polyfit(np.log(dt_values), np.log(errors1), 1)[0]
    slope2 = np.polyfit(np.log(dt_values), np.log(errors2), 1)[0]

    # --- Visualization ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Standard Linear Scale
    ax1.plot(dt_values, errors1, marker='o', color='#1f77b4', linewidth=2, label=name1)
    ax1.plot(dt_values, errors2, marker='s', color='#ff7f0e', linewidth=2, label=name2)
    ax1.set_title('Error Convergence (Linear Scale)', fontsize=13, fontweight='bold')
    ax1.set_xlabel(r'Time Step size ($\Delta t$)', fontsize=12)
    ax1.set_ylabel('Maximum Absolute Error in $I(t)$', fontsize=12)
    ax1.invert_xaxis() # dt gets smaller to the right
    ax1.legend(loc='best', fontsize=11)
    ax1.grid(True, linestyle='--', alpha=0.7)

    # Plot 2: Log-Log Scale
    ax2.loglog(dt_values, errors1, marker='o', color='#1f77b4', linewidth=2, 
               label=rf'{name1} (Slope $\approx$ {slope1:.2f})')
    ax2.loglog(dt_values, errors2, marker='s', color='#ff7f0e', linewidth=2, 
               label=rf'{name2} (Slope $\approx$ {slope2:.2f})')
    ax2.set_title('Error Convergence (Log-Log Scale)', fontsize=13, fontweight='bold')
    ax2.set_xlabel(r'Time Step size ($\Delta t$)', fontsize=12)
    ax2.invert_xaxis() # dt gets smaller to the right
    ax2.legend(loc='best', fontsize=11)
    ax2.grid(True, which="both", linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()
